In [ ]:
# Baseline model class

import pandas as pd
import os
import pickle
from sklearn import model_selection, preprocessing, metrics
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression

class TFIDFBaselineModel:
    def __init__(self, train_df: pd.DataFrame, validation_df: pd.DataFrame, text_column: str, label_column: str):
        self.train_df = train_df
        self.validation_df = validation_df
        self.text_column = text_column
        self.label_column = label_column
        self.vectorizer = TfidfVectorizer()
        self.label_encoder = LabelEncoder()
        self.model = LogisticRegression(max_iter=1000)

    def preprocess_data(self):
        # Fit the TF-IDF vectorizer on the training data and transform both training and validation data
        self.X_train = self.vectorizer.fit_transform(self.train_df[self.text_column])
        self.X_validation = self.vectorizer.transform(self.validation_df[self.text_column])

        # Encode the labels
        self.y_train = self.label_encoder.fit_transform(self.train_df[self.label_column])
        self.y_validation = self.label_encoder.transform(self.validation_df[self.label_column])

    def train_model(self):
        # Train the logistic regression model
        self.model.fit(self.X_train, self.y_train)

    def evaluate_model(self):
        # Make predictions on the validation set
        y_pred = self.model.predict(self.X_validation)

        # Calculate accuracy
        accuracy = metrics.accuracy_score(self.y_validation, y_pred)
        print(f'Validation Accuracy: {accuracy:.4f}')

    def run_pipeline(self):
        self.preprocess_data()
        self.train_model()
        self.evaluate_model()

    def save_model(self, output_dir):
        # Save the trained model, vectorizer, and label encoder to the specified output directory
        os.makedirs(output_dir, exist_ok=True)
        with open(os.path.join(output_dir, 'model.pkl'), 'wb') as f:
            pickle.dump(self.model, f)
        with open(os.path.join(output_dir, 'vectorizer.pkl'), 'wb') as f:
            pickle.dump(self.vectorizer, f)
        with open(os.path.join(output_dir, 'label_encoder.pkl'), 'wb') as f:
            pickle.dump(self.label_encoder, f)

        

In [ ]:
# BERT Model Class and TextDataset Class

import torch
import torch.nn as nn
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

class TextDataset(Dataset):

    def __init__(self, texts, labels, mode, max_len):
        self.texts = texts
        self.labels = labels
        self.encoder = LabelEncoder()
        self.tokenizer = BertTokenizer.from_pretrained(mode)
        self.max_len = max_len
        self.attention_mask = None


    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        return {
            'text': text,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

class BERTClassifier(nn.Module):
    def __init__(self, n_classes, train_loader, val_loader, pretrained_model_name='bert-base-cased'):
        super(BERTClassifier, self).__init__()
        self.bert = BertModel.from_pretrained(pretrained_model_name)
        self.drop = nn.Dropout(p=0.3)
        self.out = nn.Linear(self.bert.config.hidden_size, n_classes)
        self.train_loader = train_loader
        self.val_loader = val_loader

    def forward(self, input_ids, attention_mask):
        _, pooled_output = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        output = self.drop(pooled_output)
        return self.out(output)

    def train(self, device, optimizer, criterion, epochs):
        
        for epoch in range(epochs):
            total_loss = 0
            for batch in self.train_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)

                optimizer.zero_grad()
                outputs = self(input_ids, attention_mask)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()

                total_loss += loss.item()

            validation_loss = 0
            with torch.no_grad():
                for batch in self.val_loader:
                    input_ids = batch['input_ids'].to(device)
                    attention_mask = batch['attention_mask'].to(device)
                    labels = batch['labels'].to(device)

                    outputs = self(input_ids, attention_mask)
                    loss = criterion(outputs, labels)
                    validation_loss += loss.item()

            print(f'Epoch {epoch + 1}/{epochs}, Training Loss: {total_loss / len(self.train_loader)}, Validation Loss: {validation_loss / len(self.val_loader)}')
            
        return total_loss / len(self.train_loader), validation_loss / len(self.val_loader)


In [ ]:
# Temperature Scaling Class for Calibration Set

class TemperatureScaling(nn.Module):
    def __init__(self):
        super(TemperatureScaling, self).__init__()
        self.temperature = nn.Parameter(torch.ones(1) * 1.5)

    def forward(self, logits):
        return logits / self.temperature

    